# Notebook 2 — K-Means Clustering

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/indicium15/ml-workshop/blob/main/workshop-2/notebooks/02_KMeans_Clustering.ipynb)

**Best used for:** creating repeatable cluster assignments after choosing the number of groups.

## What is K-Means?

**K-Means** is a clustering algorithm that partitions data into *k* groups by:

1. Placing *k* **centroid** points randomly (or via k-means++ initialisation).
2. Assigning each data point to its nearest centroid.
3. Moving each centroid to the **mean** of its assigned points.
4. Repeating steps 2–3 until the assignments stop changing.

## Choosing k — The Elbow Method

K-Means requires you to specify *k* in advance. The **elbow method** helps by plotting **inertia** (total within-cluster sum of squared distances) for a range of *k* values:

- Inertia decreases as *k* increases — always.
- Look for the "elbow" where the curve bends sharply: adding more clusters beyond this point gives diminishing returns.

The **silhouette score** is a complementary measure: higher is better (range −1 to +1), measuring how well-separated the clusters are.

## Connection to HCA

In Notebook 1, we used HCA to *discover* cluster structure. Now we ask: **"Can we formalise those groupings?"** K-Means will produce hard cluster assignments that we can compare with the HCA solution.

---

## Running This Notebook

You can use this notebook in either:

- **Google Colab**: best if you do not have Python installed.
- **Local Jupyter**: best if you already have the workshop folder on your computer.

### In Google Colab

1. Open this notebook from GitHub using the **Open in Colab** button.
2. Run the first setup cell.
3. Wait until it says the libraries and workshop files are loaded.
4. In **Step 1**, either use the pre-loaded sample dataset or upload your own CSV file.
5. Continue running each cell from top to bottom.

If a widget does not appear immediately in Colab, wait for the setup cell to finish, then rerun the current cell.

In [ ]:
# SETUP
import sys, os, subprocess, importlib.util
from pathlib import Path

# Works locally from workshop-2/notebooks, from workshop-2, or in Google Colab
# after opening the notebook from GitHub.
REPO_URL = 'https://github.com/indicium15/ml-workshop.git'

def _find_workshop_root():
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path.cwd() / 'workshop-2',
        Path.cwd().parent / 'workshop-2',
        Path('/content/ml-workshop/workshop-2'),
        Path('/content/workshop-2'),
    ]
    for candidate in candidates:
        if (candidate / 'utils' / 'data_loader.py').exists():
            return candidate.resolve()
    return None

IN_COLAB = 'google.colab' in sys.modules
WORKSHOP_ROOT = _find_workshop_root()

if WORKSHOP_ROOT is None and IN_COLAB:
    print('Workshop files not found in Colab. Cloning the workshop repository...')
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, '/content/ml-workshop'], check=True)
    WORKSHOP_ROOT = _find_workshop_root()

if WORKSHOP_ROOT is None:
    raise FileNotFoundError(
        'Could not find workshop-2/utils. Locally, start Jupyter from the workshop-2 folder. '
        'In Colab, upload the workshop-2 folder or open the notebook from the GitHub repository.'
    )

if str(WORKSHOP_ROOT) not in sys.path:
    sys.path.insert(0, str(WORKSHOP_ROOT))

required_modules = {
    'numpy': 'numpy',
    'pandas': 'pandas',
    'sklearn': 'scikit-learn',
    'scipy': 'scipy',
    'matplotlib': 'matplotlib',
    'seaborn': 'seaborn',
    'ipywidgets': 'ipywidgets',
}
missing = [pip_name for module_name, pip_name in required_modules.items()
           if importlib.util.find_spec(module_name) is None]
if missing:
    print('Installing missing packages:', ', '.join(missing))
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(WORKSHOP_ROOT / 'requirements.txt')], check=True)

import ipywidgets as widgets

print(f'Using workshop files from: {WORKSHOP_ROOT}')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from IPython.display import display
import matplotlib
matplotlib.use('module://matplotlib_inline.backend_inline')
import matplotlib.pyplot as plt
%matplotlib inline

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

from utils.data_loader import DataLoaderWidget
from utils.preprocessor import PreprocessorWidget
from utils.plotting import (
    plot_elbow_curve,
    plot_silhouette,
    plot_cluster_scatter,
    plot_cluster_profile_heatmap,
)

print('Libraries loaded ✓')


### Colab Notes

- The first setup cell may take a minute because it checks for required packages.
- Uploaded files in Colab are temporary. If the runtime disconnects, upload the file again.
- To keep your own completed version, use **File → Save a copy in Drive**.
- If widgets do not display, run the setup cell again, then rerun the current cell.

## Step 1 — Load Data

The sample dataset is already available when you run the setup cell.

To use your own data in Colab:

1. Click the upload control.
2. Choose a `.csv` file from your computer.
3. Wait for the preview table to appear.
4. Select the columns you want to use.
5. Click **Confirm Selection**.

For local Jupyter, you can either upload a CSV or type a path to a CSV file on your machine.

> Keep a copy of your original dataset unchanged. These notebooks do not edit the source CSV.

> **Tip:** Use the same feature columns you used in the HCA notebook to make the comparison meaningful. Avoid ID columns and known labels when training the clustering model.

In [ ]:
# LOAD DATA
loader = DataLoaderWidget(show_label_selector=False)
loader.display()

## Step 2 — Preprocessing

> ⚠️ **Scaling is required for K-Means.** K-Means uses Euclidean distance, so features on larger scales will dominate. Always scale your data before clustering.

StandardScaler is selected and cannot be disabled.

In [ ]:
# HYPERPARAMETERS
preprocessor = PreprocessorWidget(
    source_loader=loader,
    y=None,
    force_scale=True,
    clustering_mode=True,
)
preprocessor.display()


## Step 2b — Why Normalisation Matters for K-Means

K-Means assigns every point to its **nearest centroid** using Euclidean distance.
If features are on different scales, those with the largest numerical range dominate
the distance calculation — effectively making all other features invisible.

The sample dataset contains two deliberately paired column groups:

| Without scaling — dominates | With scaling — treated equally |
|---|---|
| `total_study_minutes_per_week` (0–2400) | `avg_weekly_study_hours` (0–40) |
| `cumulative_lms_sessions_per_semester` (0–300) | `lms_logins_per_week` (0–20) |

Run the cell below to see **both** the scale chart **and** a side-by-side cluster comparison
(same *k*, same data — only scaling differs). The unscaled version will produce clusters
that are essentially just bands along the `total_study_minutes_per_week` axis.

In [ ]:
# OUTPUT
from sklearn.preprocessing import StandardScaler
from utils.plotting import plot_feature_scales, plot_normalisation_comparison

_norm_out = widgets.Output()
_norm_btn = widgets.Button(
    description='Show Normalisation Effect',
    button_style='warning',
    icon='bar-chart',
)
_norm_k = widgets.IntSlider(
    value=4, min=2, max=8, step=1,
    description='k for demo:',
    style={'description_width': '100px'},
    layout=widgets.Layout(width='320px'),
)

def _show_norm_effect(_btn):
    _norm_out.clear_output()
    with _norm_out:
        try:
            from sklearn.cluster import KMeans as _KM
            df_raw = loader.df
            if df_raw is None:
                display(widgets.HTML('<span style="color:red">Load data first.</span>'))
                return

            # ── Feature scale chart ──────────────────────────────────────────
            display(widgets.HTML('<b>Feature value ranges (raw, unscaled):</b>'))
            fig0 = plot_feature_scales(df_raw)
            plt.show()

            # ── Numeric-only subsets for the direct comparison ───────────────
            exclude = {'student_id', 'performance_band'}
            num_cols = [
                c for c in df_raw.select_dtypes(include='number').columns
                if c not in exclude
            ]
            X_raw = df_raw[num_cols].fillna(0)

            scaler = StandardScaler()
            X_scaled_arr = scaler.fit_transform(X_raw)
            import pandas as _pd
            X_scaled = _pd.DataFrame(X_scaled_arr, columns=num_cols)

            k = _norm_k.value
            km_raw = _KM(n_clusters=k, n_init=10, random_state=42).fit(X_raw)
            km_scaled = _KM(n_clusters=k, n_init=10, random_state=42).fit(X_scaled)

            display(widgets.HTML(
                f'<b>K-Means (k={k}): cluster scatter — raw vs normalised</b><br>'
                '<span style="color:#555;font-size:0.85em">Both plots show the same students '
                'projected via PCA. Only the scaling differs.</span>'
            ))
            fig1 = plot_normalisation_comparison(
                X_raw, X_scaled,
                km_raw.labels_, km_scaled.labels_,
            )
            plt.show()

            # ── Dominant feature check ───────────────────────────────────────
            import numpy as _np
            ranges = X_raw.max() - X_raw.min()
            top_feat = ranges.idxmax()
            display(widgets.HTML(
                f'<div style="background:#fff3cd;border:1px solid #ffc107;'
                f'padding:8px;border-radius:4px;margin-top:6px">'
                f'⚠ <b>Without scaling</b>, the largest-range feature '
                f'(<code>{top_feat}</code>, range = {ranges[top_feat]:.0f}) '
                f'contributes <b>{ranges[top_feat]/ranges.median():.0f}× more</b> '
                f'to Euclidean distance than the median feature (range = {ranges.median():.1f}).'
                f'</div>'
            ))

        except Exception as exc:
            import traceback
            display(widgets.HTML(f'<span style="color:red">Error: {exc}<br>'
                                 f'<pre>{traceback.format_exc()}</pre></span>'))

_norm_btn.on_click(_show_norm_effect)
display(widgets.VBox([
    _norm_k,
    _norm_btn,
    _norm_out,
]))

## Step 3 — Elbow Method

Run K-Means for a range of *k* values to find the optimal number of clusters.

Look for:
- The **elbow point** in the inertia curve (sharp bend)
- The **peak** in the silhouette score plot

In [ ]:
# HYPERPARAMETERS
_elbow_out = widgets.Output()

_k_min = widgets.IntText(value=2, description='K min:', layout=widgets.Layout(width='120px'),
                         style={'description_width': '50px'})
_k_max = widgets.IntText(value=10, description='K max:', layout=widgets.Layout(width='120px'),
                         style={'description_width': '50px'})
_elbow_btn = widgets.Button(description='Run Elbow ▶', button_style='primary', icon='play')

_inertias = []
_silhouettes = []
_k_range_used = []

def _run_elbow(_btn):
    global _inertias, _silhouettes, _k_range_used
    _elbow_out.clear_output()
    with _elbow_out:
        try:
            X_scaled = preprocessor.X_scaled
            if X_scaled is None:
                display(widgets.HTML('<span style="color:red">Run preprocessing first.</span>'))
                return

            k_min = max(2, _k_min.value)
            k_max = min(20, _k_max.value)
            k_range = range(k_min, k_max + 1)
            _k_range_used = list(k_range)

            display(widgets.HTML('Computing K-Means for each k... (this may take a moment)'))

            inertias, silhouettes = [], []
            for k in k_range:
                km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
                labels = km.fit_predict(X_scaled)
                inertias.append(km.inertia_)
                sil = silhouette_score(X_scaled, labels) if k > 1 else 0.0
                silhouettes.append(sil)

            _inertias = inertias
            _silhouettes = silhouettes

            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

            ax1.plot(_k_range_used, inertias, 'o-', color='steelblue', linewidth=2)
            ax1.set_xlabel('k'); ax1.set_ylabel('Inertia'); ax1.set_title('Elbow Curve')
            ax1.set_xticks(_k_range_used)

            ax2.plot(_k_range_used, silhouettes, 's-', color='darkorange', linewidth=2)
            best_k = _k_range_used[int(np.argmax(silhouettes))]
            ax2.axvline(x=best_k, color='red', linestyle='--', label=f'Best k={best_k}')
            ax2.set_xlabel('k'); ax2.set_ylabel('Silhouette score'); ax2.set_title('Silhouette Scores')
            ax2.set_xticks(_k_range_used)
            ax2.legend()

            plt.tight_layout()
            plt.show()
            display(widgets.HTML(
                f'<b>Suggestion:</b> Silhouette score is highest at <b>k = {best_k}</b>. '
                f'Compare with the elbow curve above.'
            ))

        except Exception as exc:
            display(widgets.HTML(f'<span style="color:red">Error: {exc}</span>'))

_elbow_btn.on_click(_run_elbow)
display(widgets.VBox([
    widgets.HTML('<h3>📊 Step 3 — Elbow Method</h3>'),
    widgets.HBox([_k_min, _k_max, _elbow_btn]),
]))
display(_elbow_out)

## Step 4 — Train K-Means

Choose your hyperparameters based on the elbow and silhouette plots above, then train the model.

In [ ]:
# HYPERPARAMETERS
_km_out = widgets.Output()
_km_model = None
_km_labels = None

_k_slider = widgets.IntSlider(
    value=4, min=2, max=10, step=1,
    description='N clusters (k):',
    style={'description_width': '150px'},
    layout=widgets.Layout(width='450px'),
)
_init_dd = widgets.Dropdown(
    options=['k-means++', 'random'],
    value='k-means++',
    description='Init method:',
    style={'description_width': '150px'},
)
_max_iter_slider = widgets.IntSlider(
    value=300, min=100, max=500, step=50,
    description='Max iterations:',
    style={'description_width': '150px'},
    layout=widgets.Layout(width='450px'),
)
_n_init_slider = widgets.IntSlider(
    value=10, min=5, max=20, step=1,
    description='N restarts:',
    style={'description_width': '150px'},
    layout=widgets.Layout(width='450px'),
)
_seed_input = widgets.IntText(
    value=42,
    description='Random state:',
    layout=widgets.Layout(width='200px'),
    style={'description_width': '110px'},
)
_train_btn = widgets.Button(
    description='Train Model ▶',
    button_style='success',
    icon='play',
)

def _train_kmeans(_btn):
    global _km_model, _km_labels
    _km_out.clear_output()
    with _km_out:
        try:
            X_scaled = preprocessor.X_scaled
            if X_scaled is None:
                display(widgets.HTML('<span style="color:red">Run preprocessing first.</span>'))
                return

            k = _k_slider.value
            _km_model = KMeans(
                n_clusters=k,
                init=_init_dd.value,
                max_iter=_max_iter_slider.value,
                n_init=_n_init_slider.value,
                random_state=_seed_input.value,
            )
            _km_labels = _km_model.fit_predict(X_scaled)
            sil = silhouette_score(X_scaled, _km_labels)

            display(widgets.HTML(
                f'<span style="color:green">✓ K-Means trained — k={k}, '
                f'inertia={_km_model.inertia_:.1f}, silhouette={sil:.4f}</span>'
            ))

            # Elbow curve with selected k highlighted
            if _inertias and _k_range_used:
                fig = plot_elbow_curve(_inertias, _k_range_used, selected_k=k)
                plt.show()

            # Silhouette plot
            fig2 = plot_silhouette(X_scaled, _km_labels,
                                   title=f'Silhouette Plot — k={k} (avg={sil:.3f})')
            plt.show()

            # 2D scatter
            fig3 = plot_cluster_scatter(
                X_scaled, _km_labels,
                method='PCA',
                title=f'K-Means Clusters — PCA projection (k={k})',
            )
            plt.show()

            # Cluster sizes
            counts = pd.Series(_km_labels).value_counts().sort_index().rename('n_students')
            display(widgets.HTML('<b>Cluster sizes:</b>'))
            display(counts.to_frame())

            # Centroid heatmap (back in original feature space)
            feat_names = list(X_scaled.columns)
            centroid_df = pd.DataFrame(
                _km_model.cluster_centers_,
                columns=feat_names,
                index=[f'Cluster {i}' for i in range(k)],
            )
            display(widgets.HTML('<b>Cluster centroids (scaled feature space):</b>'))
            fig4 = plot_cluster_profile_heatmap(centroid_df, title='Cluster Centroids')
            plt.show()

        except Exception as exc:
            display(widgets.HTML(f'<span style="color:red">Error: {exc}</span>'))

_train_btn.on_click(_train_kmeans)
display(widgets.VBox([
    widgets.HTML('<h3>🤖 Step 4 — K-Means Hyperparameters</h3>'),
    _k_slider,
    _init_dd,
    _max_iter_slider,
    _n_init_slider,
    _seed_input,
    _train_btn,
]))
display(_km_out)

## Step 5 — Compare Clusters to a Known Column

If your dataset has a known group, label, department, grade band, or other comparison column, choose it here. This is optional and is only a sanity check; K-Means did not use this column as a target.


In [ ]:
# OUTPUT
_compare_out = widgets.Output()
_compare_col = widgets.Dropdown(
    options=['(load data first)'],
    value='(load data first)',
    description='Compare with:',
    style={'description_width': '110px'},
    layout=widgets.Layout(width='420px'),
)
_refresh_compare_btn = widgets.Button(
    description='Refresh Columns',
    button_style='',
    icon='refresh',
)
_compare_btn = widgets.Button(
    description='Compare Clusters',
    button_style='info',
    icon='table',
)
_compare_note = widgets.HTML(
    '<span style="color:#555">Choose a categorical or low-cardinality column for a cross-tabulation. '
    'This helps you interpret clusters, but it is not proof that the clusters are correct.</span>'
)

def _refresh_compare_columns(_btn=None):
    if loader.df is None:
        _compare_col.options = ['(load data first)']
        _compare_col.value = '(load data first)'
        return
    feature_cols = set(loader.X_df.columns) if loader.X_df is not None else set()
    candidates = []
    for col in loader.df.columns:
        nunique = loader.df[col].nunique(dropna=True)
        if col not in feature_cols or nunique <= 30:
            candidates.append(col)
    if not candidates:
        candidates = list(loader.df.columns)
    _compare_col.options = candidates
    if 'performance_band' in candidates:
        _compare_col.value = 'performance_band'
    else:
        low_card = [c for c in candidates if loader.df[c].nunique(dropna=True) <= 30]
        _compare_col.value = low_card[0] if low_card else candidates[0]

def _compare_to_column(_btn):
    _compare_out.clear_output()
    with _compare_out:
        try:
            if _km_labels is None:
                display(widgets.HTML('<span style="color:red">Train K-Means first (Step 4).</span>'))
                return
            if loader.df is None:
                display(widgets.HTML('<span style="color:red">Load data first.</span>'))
                return
            compare_col = _compare_col.value
            if compare_col not in loader.df.columns:
                display(widgets.HTML('<span style="color:orange">Choose a comparison column and click Refresh Columns if needed.</span>'))
                return

            df_compare = pd.DataFrame({
                'cluster': _km_labels,
                compare_col: loader.df.loc[preprocessor.X_scaled.index, compare_col].astype(str).values,
            })
            cross_tab = pd.crosstab(
                df_compare['cluster'],
                df_compare[compare_col],
                margins=True,
            )
            display(widgets.HTML(f'<b>Cluster vs <code>{compare_col}</code>:</b>'))
            display(cross_tab)
            display(widgets.HTML(
                '<span style="color:#555">Rows = K-Means cluster, columns = selected comparison values. '
                'Each cell = number of records in that cluster and group.</span>'
            ))
        except Exception as exc:
            display(widgets.HTML(f'<span style="color:red">Error: {exc}</span>'))

_refresh_compare_btn.on_click(_refresh_compare_columns)
_compare_btn.on_click(_compare_to_column)
display(widgets.VBox([
    _compare_note,
    widgets.HBox([_compare_col, _refresh_compare_btn]),
    _compare_btn,
    _compare_out,
]))


## Reflection

1. **Elbow vs silhouette:** Did they suggest the same *k*? How did you resolve any disagreement?
2. **K-Means vs HCA:** Which gave you more distinct, interpretable clusters?
3. **Cross-tabulation:** Do the K-Means clusters roughly align with the performance bands? What does this tell you about the relationship between study behaviours and outcomes?
4. **Limitations:** K-Means assumes equal-sized, spherical clusters. Is that a reasonable assumption for student data?

> **Next step:** Open Notebook 3 to build a supervised model that predicts performance band directly.

---

## After the Workshop: Reusing This K-Means Notebook

Use this notebook when you want a practical clustering model that assigns every row to one of *k* groups.

K-Means is useful when:

- you already have a rough idea of how many groups you want
- you want simple cluster assignments
- you want to compare clusters against a known column
- you want a repeatable segmentation method

### What to Save

For your own analysis notes, save:

- the feature columns used
- the scaling method
- the selected value of *k*
- the elbow plot
- the silhouette scores
- the cluster size table
- the cluster profile heatmap
- any comparison table with a known column

### Reuse Warning

K-Means works best when clusters are roughly round and similarly sized. If your clusters look uneven, overlapping, or nested, compare the result with HCA before drawing conclusions.